# LLM-augmented BO: 100 spectral amplitude factors

**Random spectral mask → matched linear/nonlinear GNLSE → measured all-mode D_NL → AI chooses the next mask → repeat.**

Select **GNLSE (Python)** and run cells in order.
This follows the [phase-and-amplitude LLMBO workflow](../../Modal%20amplitudes%20and%20relative%20phases/LLM-assisted%20Bayesian%20optimization/dnl_llm_bo_phase_amplitude.ipynb),
with the launch mixture fixed and **100 spectral factors** as the optimization variables.
Implementation: [spectral_mask.py](../../../notebook_helpers/llm_bo/spectral_mask.py).

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook_helpers"))
from llm_bo.spectral_mask import run_experiment, show_results, show_timing_comparison

## 1. Settings and the 100 controls

Start from the same 200 fs Gaussian at **100 or 81.943 kW before the mask**,
with equal amplitudes in LP01, LP11-cos and LP11-sin and zero relative phases.
All three modes receive the **same mask**; modal coefficients are fixed.

Using the solver's convention,
\[
\widetilde A=\operatorname{fftshift}(\operatorname{ifft}(A)),\qquad
\widetilde A_{\rm shaped}(\omega)=s_j\widetilde A(\omega)
\quad\text{in band }j,\qquad s_j\in\{0,0.01,\ldots,1.00\}.
\]

Split the centred interval containing **99.9% of the original Gaussian spectral
energy** into 100 equal-width angular-frequency portions. That interval is fixed
before optimization. The weak tails outside it remain unchanged. An amplitude
factor of 0.5 retains 0.25 of that portion's power. **No energy or peak-power
renormalization is applied.** Inverse transformation uses
`fft(ifftshift(shaped_spectrum))`.

The old 10 ps grid cannot represent 100 occupied portions. This notebook uses
four frequency samples per portion and Nt=16384; the exact time window and band
edges are printed and saved. Final checks double the time window and halve dt
(Nt=65536), keeping the physical band edges and the same 100 factors.

The objective remains the coherent **all-three-mode D_NL** between matched
linear (`n2=0`) and nonlinear outputs. Require all-mode nonlinear output photons
to remain at least **50% of the unmasked 1 mm baseline** at the same original
power. A dim mask's small D_NL is recorded but cannot win. Attenuation and pulse
broadening are part of this search, so input energy, peak power and temporal tails
are reported alongside D_NL.

In [ ]:
POWER_KW = 100.0
MAX_TRIALS = 12           # Bounded pilot; includes the seeded random mask.
SEED = 20261113           # Reproducible, uniform random factors on the 0.01 grid.
MODEL = None  # Use the model configured in Codex.
THINKING_EFFORT = None  # Set an effort supported by your model to override.
RESUME_RUN = None  # Or supply the path to a previous run.

## 2. Run the search

### 2.1 Prepare AI and its prior

AI receives the all-mode objective, fiber, bounds, 100 band edges and initial
spectral-energy weights. It selects an executable GP prior and prints a brief
explanation. As in the previous notebook, the GP is used only for initialization;
later point selection is performed directly by AI using measured history.

### 2.2 Draw the initial random mask

Python independently draws **100 integers from 0 through 100** using `SEED` and
divides by 100. This is trial 1; AI cannot replace it. Random amplitude masks often
retain roughly one third of input energy and may fail the inherited brightness
floor. The failed constraint remains visible in the feedback.

### 2.3 Measure one mask

Apply all 100 factors in frequency space, reconstruct the complex temporal field,
and propagate that same field through the matched linear and nonlinear fibers.
One objective measurement therefore costs **two propagations**. Report measured
D_NL, output brightness, input transmission, peak power, duration and edge energy.

### 2.4 AI selects the next complete 100-factor vector

AI receives every measured mask and result, then returns one new 100-factor
vector or stops. Smooth or grouped proposals are allowed, but all 100 variables
remain independent. Python rejects missing factors, off-grid values and repeats.

The pilot has a hard limit of `MAX_TRIALS` masks. It may stop after at least eight
trials if six consecutive trials yield no feasible improvement of 0.001, or after
20 minutes of search work; final validation is retained. AI may also stop early.
Baseline and final checks add at most **five objective pairs**. With 12 trials,
the cap is **17 nonlinear + 17 linear solves and 13 AI calls**. A cached baseline
or selected mask costs no additional propagation. Failed AI attempts count toward
the cap. An AI timeout ends acquisition and retains final numerical checks.

Set `RESUME_RUN` to a saved run directory to recover an interrupted run without
resetting its budgets or repeating completed measurements. A completed directory
loads its saved result. Set it to `None` to start a new run. Active runtime includes
failed calls and recovery work, but excludes time paused between sessions.

For caps above 12, the controller uses the recommended minimum of 10 trials and
patience of 10, with a search-time cap of two minutes per allowed trial. Numerical
validation remains outside that search-time cap.

In [ ]:
# File the saved campaign under its launch controls and search method.
from notebook_helpers.run_library import file_saved_run

run = run_experiment(
    power_kW=POWER_KW,
    max_trials=MAX_TRIALS,
    seed=SEED,
    model=MODEL,
    thinking_effort=THINKING_EFFORT,
    resume_directory=RESUME_RUN,
)
file_saved_run(run.directory, "Spectral amplitude mask", "LLM-assisted Bayesian optimization")

## 3. Inspect the selected mask, pulses and numerical checks

Display the measured history, brightness and input energy, all 100 selected
factors, input/output spectra, and temporal tails. Compare the selected mask and
unmasked baseline at 5 mm, 1 mm, and on the larger time/frequency grid.
An absolute D_NL tolerance of 0.0002 is used for step and grid checks. Failed
checks leave the optical result preliminary. The table and plots below use saved
fields and launch no new simulations.

In [ ]:
show_results(run)

## 4. Estimate gradient-descent time and choose an LLMBO stopping budget

**No gradient descent, derivative probes, or line search are run here.**
The estimate uses this run's measured linear/nonlinear propagation times, setup
and I/O overhead, and AI-call times on the same machine and grids.

For 100 controls, a central finite-difference gradient costs approximately
**200 objective evaluations**, while forward/one-sided differences cost roughly
100. A trial step needs at least one more evaluation; backtracking adds more.
The table assumes a 0.01 difference step and projection onto the allowed lattice,
with a cached current score and one trial per update. It includes the same final
checks and shows 1, 10 and 20-step scenarios. A separately implemented adjoint or
autodiff method could have a different cost; it is not estimated here.

The LLMBO comparison includes its **model overhead**, which can exceed solver
time. It reports the actual pilot and estimated 30- and 60-trial caps. These are
runtime comparisons at stated budgets, not evidence of equal solution quality.

**Practical recommendation:** start with 30 trials, allow at least 10, and stop
when the last 10 trials yield no feasible D_NL improvement of 0.001. Consider a
separate extension up to 60 only if progress continues and numerical checks pass.
Resolve any grid-refinement failure first. A small run cannot establish convergence
or exhaust interactions among 100 independent factors.

In [ ]:
timing_comparison = show_timing_comparison(run)

## Replay this search

In [ ]:
from llm_bo.spectral_mask_replay import show_replay
show_replay(run)